# HiGAN+ — Recognizer-Crop Experiments on Kaggle (T4 / P100)

This notebook fine-tunes HiGAN+ from `deploy_HiGAN+.pth` under four conditions to
measure how much the recognizer-crop trick (Idea 1) loosens R's grip on G:

- **baseline** — feed each fake stream whole into R (upstream behaviour).
- **left_half** — crop the left 50% before R reads it.
- **left_three_quarter** — crop the left 75%.
- **char_aligned** — uniform random char-aligned slice; the most aggressive setting.

All four configs log FID/KID/IS to wandb every epoch via `eval_fid_every: 1`.
Set `WANDB_API_KEY` in Kaggle secrets before running, otherwise wandb silently
falls back to `mode: disabled` and only TensorBoard scalars are written.

Fine-tune HiGAN+ on the IAM word dataset starting from the upstream `deploy_HiGAN+.pth` checkpoint.

**Requirements before running:**
- Notebook *Settings* → **Accelerator: GPU T4 / P100** and **Internet: ON**.
- ~1 GB free disk for IAM hdf5 + pretrained checkpoints.

**Outputs land under `/kaggle/working/higanplus/HiGAN+/runs/`** — Kaggle persists `/kaggle/working/` as the notebook output, so checkpoints and tensorboard logs survive after the session ends.

Pipeline:
1. Clone fork → install deps (skip torch, Kaggle's image already has CUDA torch).
2. Download IAM hdf5 + 4 pretrained `.pth` from upstream releases.
3. Smoke-run (2 epochs × 30 iters) to verify the loop on Kaggle hardware.
4. Full fine-tune: 70 epochs from `gan_iam_kaggle.yml` (linear LR decay from epoch 25).

## 1. Environment check

In [ ]:
import torch, sys
print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device  :', torch.cuda.get_device_name(0))
    print('cuda ver:', torch.version.cuda)

## 2. Clone repo (fork with CUDA-12.x / PyTorch 2.x compat fixes)

In [ ]:
%cd /kaggle/working
BRANCH = 'feat/recog-random-crop'
![ -d higanplus ] || git clone --depth 1 -b $BRANCH https://github.com/ks2n/higanplus.git
%cd /kaggle/working/higanplus
# If the dir was cached from a previous run, force it onto the experiment branch.
!git fetch origin $BRANCH --depth=1 && git checkout $BRANCH && git pull --ff-only origin $BRANCH
!git log -1 --oneline


## 3. Install Python deps
Kaggle ships a CUDA-enabled torch already, so `requirements.txt` skips torch.

In [ ]:
!pip install -q -r requirements.txt

## 4. Download IAM dataset + pretrained checkpoints
Idempotent: `setup_data.sh` skips files that already exist. Total ~430 MB.

In [ ]:
!bash scripts/setup_data.sh

## 5. wandb login (optional)

With `WANDB_API_KEY` set in Kaggle Secrets, this logs in once for the whole notebook.
Without it, the trainer continues writing TensorBoard but skips wandb.

In [ ]:
import os, subprocess, sys

# Kaggle stores secrets in UserSecretsClient, not os.environ.  Try that
# first; fall back to env var so the same cell still works locally.
key = None
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret('WANDB_API_KEY')
except Exception:
    key = os.environ.get('WANDB_API_KEY')

if not key:
    print('[wandb] WANDB_API_KEY not found in Kaggle Secrets or env; runs will be local only.')
    print('         To enable: Settings -> Secrets -> Add WANDB_API_KEY, then toggle it on in this notebook (Add-ons -> Secrets).')
else:
    # Make the key visible to the wandb CLI/SDK invoked by train.py later.
    os.environ['WANDB_API_KEY'] = key
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'wandb'])
    subprocess.check_call(['wandb', 'login', '--relogin', key])
    print('[wandb] login OK; WANDB_API_KEY exported to env for child processes.')


## 6. Smoke test (verifies the crop path runs end-to-end)

This uses `gan_iam_crop_smoke.yml`: 2 epochs × 5 iters with `prob: 1.0` so every iter hits the crop code path.

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_smoke.yml


## 7. Run each experiment

Run cells one at a time on a fresh Kaggle session.  Each run takes hours; do **not** start two at once on the same instance.  Outputs land in `runs/<config-name>-<timestamp>/` with TensorBoard logs and wandb (if enabled).

> Tip: pin a different `wandb.tags` from inside the YAML if you want to re-run a variant with a tweak (e.g. different `prob`).

### 7.1. `baseline` — no crop (upstream behaviour, FID/KID still logged every epoch)

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_baseline.yml


### 7.2. `left_half` — crop left 50% before R

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_left_half.yml


### 7.3. `left_three_quarter` — crop left 75% before R

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_left_3q.yml


### 7.4. `char_aligned` — random char-aligned slice [i:j]

In [ ]:
%cd /kaggle/working/higanplus/HiGAN+
!python train.py --config ./configs/gan_iam_crop_char_aligned.yml


## 8. (Optional) Resume the latest crop run

Kaggle sessions die at 9 hours.  This cell finds the most recent `runs/gan_iam_crop_*-<timestamp>` directory and patches the matching config so the next train.py call resumes from `last.pth`.

In [ ]:
import glob, os, re
candidates = []
for stem in ('gan_iam_crop_baseline', 'gan_iam_crop_left_half', 'gan_iam_crop_left_3q', 'gan_iam_crop_char_aligned'):
    candidates += glob.glob(f'/kaggle/working/higanplus/HiGAN+/runs/{stem}-*')
candidates.sort(key=os.path.getmtime)
if not candidates:
    print('no recog-crop runs found yet')
else:
    run_dir = candidates[-1]
    last_ckpt = os.path.join(run_dir, 'ckpts', 'last.pth')
    print('latest run :', run_dir)
    print('latest ckpt:', last_ckpt, '(exists:', os.path.exists(last_ckpt), ')')
    base = os.path.basename(run_dir).rsplit('-', 4)[0]  # strip the m-d-H-M timestamp
    cfg_path = f'/kaggle/working/higanplus/HiGAN+/configs/{base}.yml'
    with open(cfg_path) as f: txt = f.read()
    new_txt = re.sub(r"pretrained_ckpt:\s*'[^']*'", f"pretrained_ckpt: '{last_ckpt}'", txt)
    if new_txt != txt:
        with open(cfg_path, 'w') as f: f.write(new_txt)
        print(f'patched {cfg_path} -> resume on next train.py call')
    else:
        print('config unchanged')


## 9. Inspect outputs

List disk usage of every recog-crop run and the latest sample images per run.

In [ ]:
!du -sh /kaggle/working/higanplus/HiGAN+/runs/gan_iam_crop_*-* 2>/dev/null || echo 'no runs yet'
import glob, os
for run in sorted(glob.glob('/kaggle/working/higanplus/HiGAN+/runs/gan_iam_crop_*-*')):
    imgs = sorted(glob.glob(os.path.join(run, 'imgs', '*.png')))
    if imgs:
        print(run, '->', os.path.basename(imgs[-1]))
